# MiniMind on Colab（T4 适配版）

仓库：`Ezra-Maker-MAX/minimind`（fork 自 `jingyaogong/minimind`，64M 参数）

本 notebook 已针对 Colab 环境规避以下实测确认的坑：

| # | 坑 | 处理 |
|---|---|---|
| 1 | `requirements.txt` 钉死 `numpy==1.26.4`，而 Colab 是 Python 3.13，该版本**没有 cp313 预编译 wheel** | 不装整份 requirements，只装训练必需子集 |
| 2 | 训练脚本默认 `--dtype bfloat16`，T4（sm_75）**无 bf16 硬件支持** | 按 GPU 架构自动切 `float16` |
| 3 | 检查点目录 `../checkpoints` 在源码里**硬编码**，落在 `/content` 下会被回收 | 软链接到 Google Drive |
| 4 | 免费层会话会被回收，mini 数据全量在 T4 上跑不完 | `--from_resume 1` 断点续训 + 缩短 epoch |

> 前置：菜单「修改 → 笔记本设置 → 硬件加速器」选 **T4 GPU** 并保存。

In [ ]:
# ── 0. 环境体检（决定 dtype，T4 必须走 float16）
import sys, torch
print('Python', sys.version.split()[0])
print('torch ', torch.__version__)
assert torch.cuda.is_available(), "没有 GPU！请先：修改 → 笔记本设置 → 硬件加速器 → T4 GPU → 保存"
p = torch.cuda.get_device_properties(0)
print('GPU   :', p.name, '| VRAM', round(p.total_memory / 1e9, 1), 'GB | sm_%d%d' % (p.major, p.minor))
bf16_ok = p.major >= 8
print('bf16 硬件支持:', bf16_ok)
DTYPE = 'bfloat16' if bf16_ok else 'float16'
print('>>> 本会话使用 dtype =', DTYPE)

In [ ]:
# ── 1. 装依赖：只装训练必需项，绕开 numpy==1.26.4 的 Python3.13 死结
#    原 requirements.txt 里 numpy==1.26.4 在 py3.13 下无 wheel（PyPI 只给 2.1.0~2.2.6）
!pip -q install "numpy>=2.1" "datasets" "transformers" "einops" "rich" "huggingface_hub"
import numpy, datasets, transformers
print('numpy', numpy.__version__, '| datasets', datasets.__version__, '| transformers', transformers.__version__)

In [ ]:
# ── 2. 挂 Drive + 把项目放 Drive
#    原因：train_pretrain.py 里 checkpoint 路径硬编码为 '../checkpoints'，
#    项目放 Drive 后它天然持久化，省掉软链，实例释放也不丢权重
from google.colab import drive
drive.mount('/content/drive')
import os

PROJ = '/content/drive/MyDrive/minimind'
!mkdir -p /content/drive/MyDrive/minimind
%cd /content/drive/MyDrive/minimind
if not os.path.exists(PROJ + '/trainer'):
    !git clone -q --depth 1 https://github.com/Ezra-Maker-MAX/minimind.git /content/mm_tmp
    !cp -r /content/mm_tmp/. /content/drive/MyDrive/minimind/
    !rm -rf /content/mm_tmp
print('项目目录:', os.getcwd())
!ls

---
## 数据集：先灌进 Drive，之后每次会话只做本地同步

体积现实（README 标称）：

| 组合 | 文件 | 体积 | 免费 Drive 15GB |
|---|---|---|---|
| **mini（推荐）** | `pretrain_t2t_mini` + `sft_t2t_mini` | **2.8GB** | ✅ |
| 完整版 | `pretrain_t2t` + `sft_t2t` | **24GB** | ❌ 放不下 |

流程：**Drive 持久化**（只下一次）→ **每次会话 copy 到 `/content`**（训练时读本地盘，
别直接读 Drive，FUSE 会把 dataloader 拖慢）。

In [ ]:
# ── 3. 一次性：把数据集灌进 Google Drive（已存在则跳过，可放心重复运行）
import os, shutil, time
from huggingface_hub import hf_hub_download

DD   = '/content/drive/MyDrive/minimind/dataset'   # 持久层
LD   = '/content/minimind/dataset'                 # 每次会话的工作层（VM 本地盘）
os.makedirs(DD, exist_ok=True); os.makedirs(LD, exist_ok=True)

def free_gb(p):
    s = os.statvfs(p); return s.f_bavail * s.f_frsize / 1e9

print('Drive 剩余 %.1f GB | 本地盘剩余 %.1f GB' % (free_gb('/content/drive'), free_gb('/content')))

FILES = ['pretrain_t2t_mini.jsonl', 'sft_t2t_mini.jsonl']   # 约 2.8GB
# 完整版是 ['pretrain_t2t.jsonl', 'sft_t2t.jsonl']，约 24GB —— 免费 Drive 装不下
if free_gb('/content/drive') < 4:
    raise SystemExit('Drive 空间不足 4GB，先清理或改用更小的数据组合')

for fn in FILES:
    dst = os.path.join(DD, fn)
    if os.path.exists(dst) and os.path.getsize(dst) > 1024:
        print('[已在 Drive]', fn, round(os.path.getsize(dst)/1e6, 1), 'MB'); continue
    t0 = time.time()
    p = hf_hub_download(repo_id='jingyaogong/minimind_dataset', filename=fn,
                        repo_type='dataset', local_dir=DD)
    sz = os.path.getsize(p) / 1e6
    print('[下载] %s %.1f MB  用时 %.0fs  (%.1f MB/s)' % (fn, sz, time.time()-t0, sz/max(time.time()-t0,1)))

In [ ]:
# ── 4. 每次会话：Drive → /content 本地盘（copy2 保留 mtime，便于 datasets 复用 arrow 缓存）
import os, shutil, time
for fn in ['pretrain_t2t_mini.jsonl', 'sft_t2t_mini.jsonl']:
    src, dst = os.path.join(DD, fn), os.path.join(LD, fn)
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
        print('[本地已有]', fn, round(os.path.getsize(dst)/1e6, 1), 'MB'); continue
    t0 = time.time(); shutil.copy2(src, dst); dt = time.time() - t0
    sz = os.path.getsize(dst) / 1e6
    print('[同步] %s %.1f MB  用时 %.0fs  (%.0f MB/s)' % (fn, sz, dt, sz/max(dt,1)))

In [ ]:
# ── 5. 冒烟测试：确认模型能实例化、参数量正确、能跑完整一步（先别烧 GPU 时长）
import sys, os, json
PROJ = '/content/drive/MyDrive/minimind'
sys.path.insert(0, PROJ)
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
from transformers import AutoTokenizer

cfg = MiniMindConfig()
model = MiniMindForCausalLM(cfg)
n = sum(q.numel() for q in model.parameters())
print('参数量: %.1fM  (README 标称 64M)' % (n / 1e6))

tok = AutoTokenizer.from_pretrained(PROJ + '/model')
print('tokenizer 词表:', len(tok))

# 造 200 行假数据走通训练流程
os.makedirs(PROJ + '/dataset', exist_ok=True)
with open(PROJ + '/dataset/smoke.jsonl', 'w', encoding='utf-8') as f:
    for i in range(200):
        f.write(json.dumps({'text': '今天天气很好，我们去公园散步吧。这是第%d条测试数据。' % i},
                           ensure_ascii=False) + '\n')
print('smoke.jsonl 就绪')

In [ ]:
# ── 6. 冒烟训练：真跑几步，验证 forward/backward/保存 全链路
!cd /content/drive/MyDrive/minimind/trainer && python train_pretrain.py \
    --epochs 1 --batch_size 8 --accumulation_steps 1 --num_workers 1 \
    --dtype {DTYPE} --data_path ../dataset/smoke.jsonl \
    --log_interval 5 --save_interval 1000

---
## 正式训练

冒烟通过后，再跑下面两格。**注意时长**：T4 约为 3090 的 40~50% 算力，
mini 数据 1 epoch 的 pretrain 预计 **2.5~3 小时**，SFT 还要更久。
Colab 免费层会话随时可能被回收，所以：

- `--from_resume 1`：断了重跑会自动接上
- 权重和检查点都落在 Drive，实例释放不丢

In [ ]:
# ── 7. 预训练（minimind-3, 64M）—— 数据读本地盘 /content/minimind/dataset
!cd /content/drive/MyDrive/minimind/trainer && python train_pretrain.py \
    --epochs 1 --batch_size 16 --accumulation_steps 8 --num_workers 2 \
    --dtype {DTYPE} --data_path /content/minimind/dataset/pretrain_t2t_mini.jsonl \
    --log_interval 50 --save_interval 500 --from_resume 1

In [ ]:
# ── 8. 监督微调（基于上一步的 pretrain 权重）
!cd /content/drive/MyDrive/minimind/trainer && python train_full_sft.py \
    --epochs 1 --batch_size 8 --accumulation_steps 2 --num_workers 2 \
    --dtype {DTYPE} --data_path /content/minimind/dataset/sft_t2t_mini.jsonl \
    --log_interval 50 --save_interval 500 --from_resume 1

In [ ]:
# ── 9. 推理验证（权重直接落在 Drive 的项目里，不会丢）
!ls -la /content/drive/MyDrive/minimind/out
!cd /content/drive/MyDrive/minimind && python eval_llm.py --weight full_sft --load_from out